# Week 1 — Data Cleaning (Yihong Yu)
**目标**：把 12 个月的原始 BTS parquet 文件清洗成一份干净的 `flights_2024_clean.parquet`

**输入**：`data/flights_2024_01.parquet` ... `data/flights_2024_12.parquet`（data_download.ipynb 产出）

**输出**：`data/flights_2024_clean.parquet`

## Step 1：挂载 Google Drive（Colab 专用）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# 修改为你实际的 Drive 路径
DATA_DIR = '/content/drive/MyDrive/CIS5450/data'
os.listdir(DATA_DIR)

## Step 2：合并 12 个月数据，只保留需要的列

In [ ]:
import pandas as pd
import numpy as np

# 只保留项目需要的列，丢弃 Div1-5 等无关字段
KEEP_COLS = [
    'FlightDate', 'Reporting_Airline', 'Tail_Number',
    'Flight_Number_Reporting_Airline',
    'Origin', 'Dest',
    'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15',
    'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15',
    'Cancelled', 'CancellationCode', 'Diverted',
    'CRSElapsedTime', 'ActualElapsedTime', 'AirTime',
    'Distance', 'TaxiOut', 'TaxiIn',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay'
]

dfs = []
for month in range(1, 13):
    path = f'{DATA_DIR}/flights_2024_{month:02d}.parquet'
    df_m = pd.read_parquet(path, columns=KEEP_COLS)
    dfs.append(df_m)
    print(f'2024-{month:02d}: {len(df_m):,} rows')

df = pd.concat(dfs, ignore_index=True)
print(f'\n合并后总行数: {len(df):,}')
print(f'列数: {len(df.columns)}')

## Step 3：过滤前50大机场

In [ ]:
TOP_50_AIRPORTS = [
    'ATL','LAX','ORD','DFW','DEN','JFK','SFO','SEA','LAS','MCO',
    'EWR','CLT','PHX','IAH','MIA','BOS','MSP','FLL','DTW','PHL',
    'LGA','BWI','SLC','DCA','SAN','MDW','TPA','HNL','PDX','STL',
    'DAL','BNA','AUS','IAD','OAK','MSY','RDU','SMF','SJC','SNA',
    'MCI','SAT','RSW','CLE','PIT','IND','CMH','OGG','ABQ','BUR'
]

before = len(df)
# Origin 或 Dest 在前50大机场内的航班都保留（保证航线完整性）
df = df[df['Origin'].isin(TOP_50_AIRPORTS) | df['Dest'].isin(TOP_50_AIRPORTS)].copy()
print(f'过滤前: {before:,} 行')
print(f'过滤后: {len(df):,} 行（减少了 {before - len(df):,} 行）')
print(f'涵盖出发机场数: {df["Origin"].nunique()}')
print(f'涵盖到达机场数: {df["Dest"].nunique()}')

## Step 3：查看数据基本情况

In [ ]:
print('=== 数据类型 ===')
print(df.dtypes)
print('\n=== 缺失值统计 ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct}).query('missing_count > 0')

In [ ]:
# 了解取消/转降航班的比例
print('取消航班数:', df['Cancelled'].sum(), f"({df['Cancelled'].mean()*100:.2f}%)")
print('转降航班数:', df['Diverted'].sum(), f"({df['Diverted'].mean()*100:.2f}%)")
print('正常运行航班数:', (df['Cancelled'] == 0).sum())

## Step 4：处理取消和转降航班

取消/转降的航班没有实际延误时间，不能用于预测建模，单独存档后从主数据移除。

In [ ]:
# 单独保存取消航班（供 EDA 分析用）
df_cancelled = df[df['Cancelled'] == 1].copy()
df_cancelled.to_parquet(f'{DATA_DIR}/flights_2024_cancelled.parquet', index=False)
print(f'取消航班已保存: {len(df_cancelled):,} 行')

# 主数据只保留正常运行的航班
df = df[(df['Cancelled'] == 0) & (df['Diverted'] == 0)].copy()
print(f'过滤后主数据: {len(df):,} 行')

## Step 5：处理关键字段的缺失值

In [ ]:
before = len(df)

# DepDelay / DepDel15 缺失 → 无法建模，直接删除
df = df.dropna(subset=['DepDelay', 'DepDel15', 'ArrDelay'])
print(f'删除 DepDelay/ArrDelay 缺失行: {before - len(df):,} 行，剩余 {len(df):,} 行')

# 延误原因字段（CarrierDelay 等）只在延误>15min 时填充，其余为 NaN 属正常，填 0
delay_cause_cols = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
df[delay_cause_cols] = df[delay_cause_cols].fillna(0)

# Tail_Number 缺失 → 填 'UNKNOWN'（不影响其他特征，连锁特征时会自然过滤）
df['Tail_Number'] = df['Tail_Number'].fillna('UNKNOWN')

print('延误原因字段和 Tail_Number 处理完毕')

## Step 6：处理异常值

In [ ]:
# 查看延误分布
print('DepDelay 分布:')
print(df['DepDelay'].describe(percentiles=[.90, .95, .99, .999]))

# 延误超过 600 分钟（10小时）视为极端异常，记录后移除
extreme_delay = df[df['DepDelay'] > 600]
print(f'\n延误 > 600 分钟的航班: {len(extreme_delay):,} 行')
extreme_delay.to_parquet(f'{DATA_DIR}/flights_2024_extreme.parquet', index=False)

before = len(df)
df = df[df['DepDelay'] <= 600].copy()
print(f'移除后剩余: {len(df):,} 行（移除了 {before - len(df):,} 行）')

In [ ]:
# 检查时间字段合理性（CRSDepTime 应在 0000-2359 之间）
invalid_time = df[(df['CRSDepTime'] < 0) | (df['CRSDepTime'] > 2359)]
print(f'CRSDepTime 异常值: {len(invalid_time)} 行')
if len(invalid_time) > 0:
    df = df[(df['CRSDepTime'] >= 0) & (df['CRSDepTime'] <= 2359)].copy()

## Step 7：类型转换与字段规范化

In [ ]:
# 日期字段转 datetime
df['FlightDate'] = pd.to_datetime(df['FlightDate'])

# 数值字段确保正确类型
int_cols = ['CRSDepTime', 'CRSArrTime', 'Distance']
df[int_cols] = df[int_cols].astype(int)

# 分类字段转 category（节省内存）
cat_cols = ['Reporting_Airline', 'Origin', 'Dest', 'CancellationCode']
df[cat_cols] = df[cat_cols].astype('category')

# DepDel15 确保是 int
df['DepDel15'] = df['DepDel15'].astype(int)

print('类型转换完成')
print(df.dtypes)

## Step 8：清洗质量报告

In [ ]:
print('=' * 50)
print('清洗质量报告')
print('=' * 50)
print(f'原始总行数:      7,079,061')
print(f'清洗后行数:      {len(df):,}')
print(f'保留比例:        {len(df)/7079061*100:.1f}%')
print(f'列数:            {len(df.columns)}')
print(f'延误率(>15min):  {df["DepDel15"].mean()*100:.2f}%')
print(f'平均延误时间:    {df["DepDelay"].mean():.2f} 分钟')
print(f'日期范围:        {df["FlightDate"].min()} ~ {df["FlightDate"].max()}')
print(f'剩余缺失值:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## Step 9：保存清洗结果

In [ ]:
output_path = f'{DATA_DIR}/flights_2024_clean.parquet'
df.to_parquet(output_path, index=False)
print(f'已保存到: {output_path}')
print(f'文件大小: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB')